# Aula 07 - Aprendizado Supervisionado parte II

**Módulo 03 IN** - Lógica para predição com inteligência artificial
**04/09/2026 - Sprint 3 - Prof. Ovidio Lopes da Cruz Netto**

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/canaldoovidio/2026-2A-M03/blob/main/notebooks/aula07.ipynb)

## O que este notebook é

A Aula 05 ajustou uma regressão linear para o Modelo 1 do TAPI e mediu MAPE de 4,25% contra a
baseline de coeficiente fixo da LDC, com 3,71%. Este notebook testa se um modelo mais flexível,
a árvore de decisão, supera essa baseline. A base usada aqui é mensal, não trimestral: as cinco
séries do SIDRA trazem a decomposição por mês, e a base mensal dá 339 observações, contra as 113
da base trimestral usada nas Aulas 04 e 05, o suficiente para treinar uma árvore sem depender de
poucos exemplos por folha.

Quatro hipóteses estruturam o notebook, cada uma fechada por uma medição:

- H1: a granularidade trimestral do case é imposta pela fonte de dados.
- H2: a árvore de decisão captura algo que a reta não captura, e por isso vence.
- H3: quando a árvore perde, a causa é a falta de extrapolação de tendência.
- H4: trocar o alvo de nível para razão sobre o mesmo mês do ano anterior devolve a árvore à
  disputa.

## Ao final deste notebook você terá

1. montado a base analítica mensal, com as defasagens de 1, 2, 3 e 12 meses e o par seno/cosseno
   de sazonalidade;
2. conferido que a soma dos três meses de cada trimestre reconcilia com a base trimestral já
   publicada;
3. comparado `DecisionTreeRegressor`, `LinearRegression` e três baselines sobre o mesmo teste;
4. lido as 24 previsões da árvore no período de teste e identificado o teto que ela não
   ultrapassa;
5. comparado KNN padronizado e não padronizado, e medido a participação de cada variável na
   distância;
6. treinado os mesmos modelos sobre o alvo em razão e fechado com o modelo que bate a baseline
   da LDC.

## 1. A base mensal

A célula abaixo lê as cinco séries de `dados/mensal/` (abate de bovinos, suínos e frangos,
produção de ovos e de leite), junta pela coluna `periodo` e monta a base analítica que o resto
do notebook usa. A definição é a mesma registrada no projeto da aula, reimplementada aqui porque
o notebook precisa rodar sozinho, sem depender de nenhum outro arquivo do repositório:

- junção interna das cinco séries por `periodo`, em ordem cronológica;
- `mes` é o inteiro dos dois últimos caracteres de `periodo`; `dias` é o número de dias do mês
  civil;
- `sen` e `cos` são o par de sazonalidade, `sin(2*pi*mes/12)` e `cos(2*pi*mes/12)`;
- `lag1`, `lag2`, `lag3` e `lag12` são `abate_frangos` defasado em 1, 2, 3 e 12 meses;
- `<serie>_lag1` são as outras quatro séries defasadas em 1 mês;
- linhas com qualquer valor ausente são descartadas, o que sacrifica os primeiros 12 meses da
  série (por causa de `lag12`).

O alvo do notebook inteiro é `abate_frangos`, o mesmo das Aulas 04 e 05. `FEATURES` fica restrito
a `lag1`, `lag12`, `sen` e `cos` até a seção 6, onde o modelo do fecho usa um conjunto maior. Os
últimos 24 meses (2024-04 a 2026-03) formam o teste: o mesmo horizonte de 24 meses que o TAPI
pede como granularidade final de entrega.

A célula traz um `try`/`except` para o caso de a rede da sala cair no meio da leitura: se a
internet falhar e o arquivo não estiver na pasta local, a mensagem de erro orienta a pedir a
pasta `dados` para uma dupla vizinha.

In [1]:
import calendar
import os
import urllib.request

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor

SERIES = [
    "abate_bovinos",
    "abate_suinos",
    "abate_frangos",
    "producao_ovos",
    "producao_leite",
]
ALVO = "abate_frangos"
FEATURES = ["lag1", "lag12", "sen", "cos"]
N_TESTE = 24
SEMENTE = 42

# mesma resolucao de caminho das aulas anteriores: funciona no repositorio
# clonado (CSVs em ../dados/mensal/) e no Colab (baixa da versao publicada)
BASE_LOCAL = os.path.join("..", "dados", "mensal")
BASE_BRUTA = ("https://raw.githubusercontent.com/canaldoovidio/2026-2A-M03/"
              "main/dados/mensal/")

caminhos = {}
for nome in SERIES:
    arquivo = nome + ".csv"
    local = os.path.join(BASE_LOCAL, arquivo)
    if os.path.exists(local):
        caminhos[nome] = local
    else:
        if not os.path.exists(arquivo):
            try:
                urllib.request.urlretrieve(BASE_BRUTA + arquivo, arquivo)
            except Exception as erro:
                raise RuntimeError(
                    "Nao foi possivel baixar '%s' pela internet (%s). "
                    "Se a rede da sala falhou, peca a pasta 'dados' para uma dupla "
                    "que tenha o repositorio clonado no computador (ela fica na raiz "
                    "do repositorio) e coloque essa pasta ao lado deste notebook. "
                    "Depois, rode esta celula de novo." % (arquivo, erro)
                ) from erro
        caminhos[nome] = arquivo

# junta as cinco series por periodo (interseccao, mesma regra da Aula 04)
base = None
for nome in SERIES:
    coluna = (pd.read_csv(caminhos[nome])[["periodo", "valor"]]
              .rename(columns={"valor": nome}))
    base = coluna if base is None else base.merge(coluna, on="periodo", how="inner")
base = base.sort_values("periodo").reset_index(drop=True)

base["mes"] = base["periodo"].str[-2:].astype(int)
base["dias"] = [calendar.monthrange(int(p[:4]), int(p[-2:]))[1] for p in base["periodo"]]
base["sen"] = np.sin(2 * np.pi * base["mes"] / 12)
base["cos"] = np.cos(2 * np.pi * base["mes"] / 12)

# defasagens do alvo: o passado entra na linha do presente
for k in (1, 2, 3, 12):
    base["lag%d" % k] = base[ALVO].shift(k)
# defasagem de 1 mes das outras quatro series
for nome in SERIES:
    if nome != ALVO:
        base[nome + "_lag1"] = base[nome].shift(1)

base = base.dropna().reset_index(drop=True)

print("base.shape:", base.shape)
print("primeiro periodo:", base["periodo"].iloc[0])
print("ultimo periodo:", base["periodo"].iloc[-1])

CORTE = len(base) - N_TESTE
print()
print("treino: %d meses (%s a %s)" % (
    CORTE, base["periodo"].iloc[0], base["periodo"].iloc[CORTE - 1]))
print("teste: %d meses (%s a %s)" % (
    N_TESTE, base["periodo"].iloc[CORTE], base["periodo"].iloc[-1]))

base.shape: (339, 18)
primeiro periodo: 1998-01
ultimo periodo: 2026-03

treino: 315 meses (1998-01 a 2024-03)
teste: 24 meses (2024-04 a 2026-03)


A base sai com 339 linhas, de 1998-01 a 2026-03. Os primeiros 12 meses de dado bruto (1997)
saem porque `lag12` só existe a partir do décimo terceiro mês da série. O corte de treino e teste
fica em 315 meses de treino (1998-01 a 2024-03) e 24 meses de teste (2024-04 a 2026-03), sempre
por data, nunca por sorteio: a Aula 05 já mostrou que embaralhar antes de separar deixa
informação do futuro vazar para o treino.

## 2. H1, a reconciliação

A primeira hipótese testa a origem da granularidade, antes de qualquer modelo entrar em
cena. A Aula 02 (CRISP-DM) registrou que o TAPI pede previsão mensal e a base das Aulas 04 e 05 é trimestral, e deixou em aberto se
essa granularidade era uma imposição da fonte ou uma escolha do projeto. A base mensal que acabou
de ser montada responde a pergunta: o SIDRA tem, sim, decomposição por mês, na mesma tabela que
já fornecia o trimestre.

A célula abaixo confere isso de forma direta, sem depender de leitura de documentação: soma os
três meses de cada trimestre das três séries de abate e compara contra os valores trimestrais já
publicados em `dados/`. Se a soma não bater com o trimestre, o mapeamento de mês para trimestre
estaria errado, e a série mensal estaria deslocada no tempo.

In [2]:
SERIES_ABATE = ["abate_bovinos", "abate_suinos", "abate_frangos"]

# mesma resolucao de caminho, agora para a pasta trimestral dados/
BASE_LOCAL_TRIMESTRAL = os.path.join("..", "dados")
BASE_BRUTA_TRIMESTRAL = "https://raw.githubusercontent.com/canaldoovidio/2026-2A-M03/main/dados/"

caminhos_trimestrais = {}
for nome in SERIES_ABATE:
    arquivo = nome + ".csv"
    local = os.path.join(BASE_LOCAL_TRIMESTRAL, arquivo)
    if os.path.exists(local):
        caminhos_trimestrais[nome] = local
    else:
        if not os.path.exists(arquivo):
            try:
                urllib.request.urlretrieve(BASE_BRUTA_TRIMESTRAL + arquivo, arquivo)
            except Exception as erro:
                raise RuntimeError(
                    "Nao foi possivel baixar '%s' pela internet (%s). "
                    "Se a rede da sala falhou, peca a pasta 'dados' para uma dupla "
                    "que tenha o repositorio clonado no computador (ela fica na raiz "
                    "do repositorio) e coloque essa pasta ao lado deste notebook. "
                    "Depois, rode esta celula de novo." % (arquivo, erro)
                ) from erro
        caminhos_trimestrais[nome] = arquivo

print("%-16s %14s %14s" % ("serie", "trimestres", "divergentes"))
divergencias_totais = 0
trimestres_conferidos = 0
for nome in SERIES_ABATE:
    trimestral = pd.read_csv(caminhos_trimestrais[nome])
    trimestral = dict(zip(trimestral["periodo"], trimestral["valor"].astype(float)))

    mensal_serie = pd.read_csv(caminhos[nome])
    soma = {}
    for _, linha in mensal_serie.iterrows():
        ano, mes = linha["periodo"].split("-")
        chave = "%s-T%d" % (ano, (int(mes) - 1) // 3 + 1)
        soma[chave] = soma.get(chave, 0.0) + float(linha["valor"])

    comuns = sorted(set(trimestral) & set(soma))
    divergentes = [c for c in comuns if trimestral[c] != soma[c]]
    divergencias_totais += len(divergentes)
    trimestres_conferidos = len(comuns)
    print("%-16s %14d %14d" % (nome, len(comuns), len(divergentes)))

print()
print("divergencia total nas tres series de abate: %d" % divergencias_totais)

serie                trimestres    divergentes
abate_bovinos               117              0
abate_suinos                117              0
abate_frangos               117              0

divergencia total nas tres series de abate: 0


A soma dos três meses reconcilia com o trimestre já publicado nas três séries de abate, sem
nenhuma divergência: zero trimestres divergentes em cada uma delas. H1 é falsa. O SIDRA sempre
teve o mês disponível, na mesma tabela que já fornecia o trimestre. A granularidade trimestral
das Aulas 04 e 05 foi uma escolha do projeto, registrada em ADR, e o autoestudo de hoje justifica
por que essa decisão continua valendo mesmo agora que a base mensal existe.

## 3. H2, a árvore perde

`DecisionTreeRegressor` particiona o espaço de entradas em regiões e prevê, para qualquer ponto
de uma região, a média do alvo observada ali no treino. Cada partição corta uma variável em um
limiar, o limiar que mais reduz o erro quadrático dentro dos dois pedaços resultantes. Com
`max_depth=3`, a árvore desta seção faz no máximo três cortes em sequência antes de virar folha,
oito folhas no total.

A hipótese H2 diz que essa flexibilidade (a árvore aprende cortes não lineares que a reta não
aprende) é suficiente para vencer a regressão linear da Aula 05 e a baseline de coeficiente fixo
da LDC. A célula abaixo treina a árvore, a mesma reta padronizada da Aula 05 e as três baselines
sobre a base mensal, todas no alvo em nível, e mede RMSE e MAPE das quatro sobre os mesmos 24
meses de teste.

In [3]:
def fatiar(features, alvo_em_razao=False):
    """Separa treino e teste por data (sem sorteio) e devolve X, alvo de treino,
    X de teste, alvo real de teste em nivel e o lag12 do teste (para reconverter
    previsoes em razao de volta a nivel)."""
    X = base[features].to_numpy()
    y = base[ALVO].to_numpy()
    lag12 = base["lag12"].to_numpy()
    alvo = (y / lag12) if alvo_em_razao else y
    return X[:CORTE], alvo[:CORTE], X[CORTE:], y[CORTE:], lag12[CORTE:]


def ajustar(features, modelo, alvo_em_razao=False, padronizar=False):
    Xtr, alvo_tr, Xte, yte, lag12te = fatiar(features, alvo_em_razao)
    if padronizar:
        escalador = StandardScaler().fit(Xtr)
        Xtr, Xte = escalador.transform(Xtr), escalador.transform(Xte)
    previsto = modelo.fit(Xtr, alvo_tr).predict(Xte)
    if alvo_em_razao:
        previsto = previsto * lag12te
    return yte, previsto


def mape(real, previsto):
    return float(np.mean(np.abs((real - previsto) / real)) * 100)


def rmse(real, previsto):
    return float(np.sqrt(np.mean((real - previsto) ** 2)))


def baselines():
    lag1 = base["lag1"].to_numpy()
    lag12 = base["lag12"].to_numpy()
    y = base[ALVO].to_numpy()
    # fator fixo: media, no treino, da razao entre o alvo e o mesmo mes do ano anterior
    fator = float(np.mean(y[:CORTE] / lag12[:CORTE]))
    return {
        "A, repete o mes anterior": lag1[CORTE:],
        "B, mesmo mes do ano anterior": lag12[CORTE:],
        "C, coeficiente fixo da LDC": lag12[CORTE:] * fator,
    }, fator


linhas_baseline, fator = baselines()
yte = base[ALVO].to_numpy()[CORTE:]

_, p_arvore = ajustar(FEATURES, DecisionTreeRegressor(max_depth=3, random_state=SEMENTE))
_, p_reta = ajustar(FEATURES, LinearRegression(), padronizar=True)

print("fator da baseline C: %.5f" % fator)
print()
print("%-32s %16s %10s" % ("modelo", "RMSE", "MAPE"))
resultados_h2 = list(linhas_baseline.items()) + [
    ("regressao linear (nivel)", p_reta),
    ("arvore de decisao (nivel)", p_arvore),
]
for nome, previsto in resultados_h2:
    print("%-32s %16.0f %9.2f%%" % (nome, rmse(yte, previsto), mape(yte, previsto)))

fator da baseline C: 1.05299

modelo                                       RMSE       MAPE
A, repete o mes anterior                 88925206      6.73%
B, mesmo mes do ano anterior             75627562      5.10%
C, coeficiente fixo da LDC               54523588      3.71%
regressao linear (nivel)                 63225293      4.25%
arvore de decisao (nivel)               109534336      7.66%


A árvore fica em MAPE de 7,66%, a pior das quatro. A reta chega a 4,25%, e a melhor baseline (C,
o coeficiente fixo que a LDC usa hoje) chega a 3,71%. H2 é falsa: a árvore não vence, nem a reta,
nem a baseline mais simples das três. A seção 4 mede por que a árvore fica tão atrás.

## 4. H3, o teto

A célula abaixo imprime a previsão da mesma árvore para cada um dos 24 meses do período de
teste, mês a mês, antes de qualquer explicação.

In [4]:
Xtr, ytr, Xte, yte, _ = fatiar(FEATURES, alvo_em_razao=False)
arvore = DecisionTreeRegressor(max_depth=3, random_state=SEMENTE).fit(Xtr, ytr)
previsto = arvore.predict(Xte)
periodos_teste = base["periodo"].to_numpy()[CORTE:]

print("previsao da arvore, mes a mes, no periodo de teste:")
for p, v in zip(periodos_teste, previsto):
    print("  %s   %.0f" % (p, v))

valores_distintos = np.unique(previsto)
print()
print("previsoes distintas emitidas nos 24 meses: %d" % len(valores_distintos))

previsao da arvore, mes a mes, no periodo de teste:
  2024-04   1090166234
  2024-05   1090166234
  2024-06   1090166234
  2024-07   1090166234
  2024-08   1090166234
  2024-09   1090166234
  2024-10   1090166234
  2024-11   1090166234
  2024-12   1090166234
  2025-01   1090166234
  2025-02   1090166234
  2025-03   1090166234
  2025-04   1090166234
  2025-05   1090166234
  2025-06   1090166234
  2025-07   1090166234
  2025-08   1090166234
  2025-09   1090166234
  2025-10   1090166234
  2025-11   1090166234
  2025-12   1090166234
  2026-01   1090166234
  2026-02   1090166234
  2026-03   1090166234

previsoes distintas emitidas nos 24 meses: 1


A coluna de previsão não muda: a árvore emite o mesmo número, 1.090.166.234, nos 24 meses de
teste. O mesmo valor de ponto flutuante se repete nas 24 linhas, sem nenhum arredondamento de
exibição envolvido. Isso é consequência direta de como a árvore funciona: com `max_depth=3`
ela tem no máximo oito folhas, e cada folha prevê a média do alvo observada ali no treino. Se os 24 meses de teste caem
todos na mesma folha, a previsão dos 24 é, necessariamente, a mesma média.

A célula seguinte confirma isso e mede a distância entre essa previsão fixa e o que de fato
aconteceu no período de teste.

In [5]:
teto = float(previsto.max())
media_real = float(yte.mean())
diferenca = (media_real - teto) / media_real * 100

print("numero de folhas da arvore: %d" % arvore.get_n_leaves())
print("teto da arvore (maior previsao possivel): %.0f" % teto)
print("media real do periodo de teste: %.0f" % media_real)
print("o valor unico da arvore fica %.2f%% abaixo da media real do teste" % diferenca)

acima_teto = int((yte > teto).sum())
acima_treino = int((yte > ytr.max()).sum())
idx_max_treino = int(np.argmax(ytr))
periodo_max_treino = base["periodo"].to_numpy()[:CORTE][idx_max_treino]
idx_max_teste = int(np.argmax(yte))
periodo_max_teste = periodos_teste[idx_max_teste]

print()
print("meses de teste acima do teto: %d de %d" % (acima_teto, N_TESTE))
print("maximo do alvo no treino: %.0f (%s)" % (float(ytr.max()), periodo_max_treino))
print("meses de teste acima do maximo do treino: %d de %d" % (acima_treino, N_TESTE))
print("maximo real no teste: %.0f (%s)" % (float(yte.max()), periodo_max_teste))

numero de folhas da arvore: 8
teto da arvore (maior previsao possivel): 1090166234
media real do periodo de teste: 1181946133
o valor unico da arvore fica 7.77% abaixo da media real do teste

meses de teste acima do teto: 23 de 24
maximo do alvo no treino: 1226709256 (2023-03)
meses de teste acima do maximo do treino: 5 de 24
maximo real no teste: 1301022625 (2026-03)


O teto da árvore, 1.090.166.234, é a média da folha mais alta que a árvore conseguiu formar
dentro do treino: ela não tem como prever um número maior do que isso, porque toda previsão é a
média de alguma das oito folhas. `abate_frangos` continua crescendo depois de 2024, e 23 dos 24
meses de teste ficam acima desse teto, o que faz a árvore errar sistematicamente para baixo em
quase todo o período. Mesmo o próprio treino já tinha visto valores maiores: 5 dos 24 meses de
teste superam até o máximo observado no treino, em 2023-03.

H3 é verdadeira: a árvore perde da reta e da baseline porque não extrapola tendência. A regressão
linear, ao contrário, não tem esse teto: uma reta se estende para qualquer valor de entrada, para
cima ou para baixo, o que já era visível na Aula 05 quando `producao_leite` entrou como preditora
fora do intervalo de treino.

## 5. KNN e a distância

`KNeighborsRegressor` prevê a média do alvo dos `k` pontos de treino mais próximos do ponto que
se quer prever, e "mais próximo" é definido pela distância euclidiana entre as colunas de
entrada. A Aula 05 já tinha estabelecido que padronizar (subtrair a média e dividir pelo
desvio-padrão de cada coluna) é necessário sempre que a distância entre colunas de escalas
diferentes importa para o modelo. A célula abaixo testa se essa mesma regra vale para o KNN nesta
base, em três valores de `k`.

In [6]:
print("%-6s %18s %18s" % ("k", "padronizado", "sem padronizar"))
for k in (3, 5, 10):
    _, p_padronizado = ajustar(FEATURES, KNeighborsRegressor(n_neighbors=k), padronizar=True)
    _, p_sem_padronizar = ajustar(FEATURES, KNeighborsRegressor(n_neighbors=k), padronizar=False)
    print("%-6d %17.2f%% %17.2f%%" % (
        k, mape(yte, p_padronizado), mape(yte, p_sem_padronizar)))

k             padronizado     sem padronizar
3                   8.14%              5.82%
5                   9.37%              5.62%
10                 10.09%              6.10%


Nos três valores de `k`, o KNN sem padronizar tem MAPE menor do que o KNN padronizado: em
`k=5`, 5,62% contra 9,37%. É o oposto do que a Aula 05 ensinou para a regressão linear.
A explicação não está no modelo, está na composição das quatro colunas de entrada: `lag1` e
`lag12` são valores de produção, na casa das centenas de milhões, e `sen` e `cos` são números
entre -1 e 1. Sem padronizar, a distância euclidiana é dominada inteiramente pelas duas colunas
de maior escala.

In [7]:
Xtr_sem_padronizar, _, _, _, _ = fatiar(FEATURES, alvo_em_razao=False)
variancias = Xtr_sem_padronizar.var(axis=0)
participacao = variancias / variancias.sum() * 100

print("participacao de cada variavel na distancia, sem padronizar:")
for nome, p in zip(FEATURES, participacao):
    print("  %-6s %8.4f%%" % (nome, p))

participacao de cada variavel na distancia, sem padronizar:
  lag1    49.2775%
  lag12   50.7225%
  sen      0.0000%
  cos      0.0000%


`lag1` responde por 49,28% da distância e `lag12` por 50,72%: juntas, praticamente 100%. `sen`
e `cos` não participam, a participação de cada um fica abaixo de 0,0001%, porque a variância de
uma coluna entre -1 e 1 é desprezível diante da variância de uma coluna na casa das centenas de
milhões de quilogramas. Sem padronizar, o KNN escolhe os vizinhos praticamente só pela produção
recente, ignorando a sazonalidade, o que aqui ajuda: `lag1` e `lag12` já carregam a informação
mais relevante para prever o próximo mês. A regra da Aula 05 continua valendo como princípio:
a escala de cada variável só deveria participar da distância quando isso melhora o modelo, e a
resposta muda de base para base. Nesta base e neste conjunto de entradas, deixá-la participar
piora o modelo.

## 6. H4, o alvo em razão

A última hipótese testa uma mudança no alvo que cada modelo aprende, com os mesmos
modelos das seções anteriores. Em vez de prever `abate_frangos` em nível, cada modelo passa a prever a razão entre o alvo e `lag12`, o mesmo mês
do ano anterior, e a previsão final é essa razão multiplicada de volta pelo `lag12` do mês
correspondente no teste. A ideia testa diretamente o motivo encontrado em H3: se a árvore perde
porque não extrapola nível, prever uma razão (que cresce muito menos do que o nível ao longo do
tempo) deveria devolvê-la à disputa.

A célula treina árvore, floresta e KNN sobre o alvo em razão, e também a reta, para checar se o
ganho é geral ou específico dos modelos que não extrapolam.

In [8]:
_, p_arvore_razao = ajustar(
    FEATURES, DecisionTreeRegressor(max_depth=3, random_state=SEMENTE), alvo_em_razao=True)
_, p_floresta_razao = ajustar(
    FEATURES, RandomForestRegressor(n_estimators=300, random_state=SEMENTE), alvo_em_razao=True)
_, p_knn_padronizado_razao = ajustar(
    FEATURES, KNeighborsRegressor(n_neighbors=5), alvo_em_razao=True, padronizar=True)
_, p_knn_sem_padronizar_razao = ajustar(
    FEATURES, KNeighborsRegressor(n_neighbors=5), alvo_em_razao=True, padronizar=False)
_, p_reta_razao = ajustar(
    FEATURES, LinearRegression(), alvo_em_razao=True, padronizar=True)

print("%-34s %16s %10s" % ("modelo (alvo em razao)", "RMSE", "MAPE"))
resultados_h4 = [
    ("arvore de decisao", p_arvore_razao),
    ("random forest (300 arvores)", p_floresta_razao),
    ("knn k=5 padronizado", p_knn_padronizado_razao),
    ("knn k=5 sem padronizar", p_knn_sem_padronizar_razao),
    ("regressao linear", p_reta_razao),
]
for nome, previsto in resultados_h4:
    print("%-34s %16.0f %9.2f%%" % (nome, rmse(yte, previsto), mape(yte, previsto)))

print()
print("arvore, nivel:  MAPE %.2f%%" % mape(yte, p_arvore))
print("arvore, razao:  MAPE %.2f%%" % mape(yte, p_arvore_razao))

modelo (alvo em razao)                         RMSE       MAPE
arvore de decisao                          59186695      3.86%
random forest (300 arvores)                70292218      4.48%
knn k=5 padronizado                        69920366      4.59%
knn k=5 sem padronizar                     56355917      3.71%
regressao linear                           71970145      4.73%

arvore, nivel:  MAPE 7.66%
arvore, razao:  MAPE 3.86%


A árvore cai de 7,66% para 3,86% de MAPE, o que a coloca à frente da regressão linear em nível
(4,25%) e quase empatada com a baseline C (3,71%). O `RandomForestRegressor`, uma média de 300
árvores independentes que reduz a variância de qualquer árvore isolada sem mudar o teto de cada
uma, melhora sobre a árvore única mas também fica atrás da baseline. A reta, ao contrário, piora
com o alvo em razão (a razão tira dela justamente a informação de nível que a reta já usava bem)
o que impede a leitura de que alvo em razão é sempre melhor: o ganho depende de o modelo não
extrapolar tendência, e a reta já extrapolava.

H4 é verdadeira: alvo em razão devolve a árvore à disputa, mas não a torna, sozinha, melhor do
que a baseline da LDC. A última célula desta seção testa se um modelo maior, com mais
informação, consegue o que nenhum dos modelos anteriores conseguiu: bater a baseline C de forma
clara.

In [9]:
FEATURES_FECHO = (
    ["lag1", "lag2", "lag3", "lag12", "sen", "cos", "dias"]
    + [serie + "_lag1" for serie in SERIES if serie != ALVO]
)

_, p_fecho = ajustar(FEATURES_FECHO, LinearRegression(), alvo_em_razao=True, padronizar=True)

print("modelo do fecho: regressao linear, alvo em razao, %d features" % len(FEATURES_FECHO))
print("features:", FEATURES_FECHO)
print()
print("%-32s %16s %10s" % ("modelo", "RMSE", "MAPE"))
print("%-32s %16.0f %9.2f%%" % (
    "baseline C (coeficiente fixo)", rmse(yte, linhas_baseline["C, coeficiente fixo da LDC"]),
    mape(yte, linhas_baseline["C, coeficiente fixo da LDC"])))
print("%-32s %16.0f %9.2f%%" % ("modelo do fecho", rmse(yte, p_fecho), mape(yte, p_fecho)))

modelo do fecho: regressao linear, alvo em razao, 11 features
features: ['lag1', 'lag2', 'lag3', 'lag12', 'sen', 'cos', 'dias', 'abate_bovinos_lag1', 'abate_suinos_lag1', 'producao_ovos_lag1', 'producao_leite_lag1']

modelo                                       RMSE       MAPE
baseline C (coeficiente fixo)            54523588      3.71%
modelo do fecho                          46724434      3.32%


O modelo do fecho junta as defasagens de 1, 2 e 3 meses, a defasagem de 12 meses, o par de
sazonalidade, os dias do mês e a defasagem de 1 mês das outras quatro séries: onze entradas, uma
regressão linear só, treinada sobre o alvo em razão. O MAPE fica em 3,32%, abaixo dos 3,71% da
baseline da LDC, com RMSE de 46.724.434 contra 54.523.588. É o primeiro modelo do notebook que
bate a baseline de forma clara, e faz isso combinando as duas descobertas da aula: alvo em razão
(H4) e mais informação de entrada do que as quatro features usadas até aqui.

## O quadro das quatro hipóteses

A célula abaixo reúne o veredito e a evidência medida de cada hipótese. É o que a dupla leva
para a ART.6.

In [10]:
quadro = [
    ("H1", "A granularidade do case e trimestral porque a fonte so tem trimestre",
     "falsa",
     "a soma dos tres meses de cada trimestre reconcilia sem divergencia nas tres "
     "series de abate: %d trimestres conferidos por serie, %d divergentes no total" % (
         trimestres_conferidos, divergencias_totais)),
    ("H2", "A arvore captura o que a reta nao captura, logo vence",
     "falsa",
     "MAPE %.2f%% da arvore contra %.2f%% da reta e %.2f%% da baseline C" % (
         mape(yte, p_arvore), mape(yte, p_reta),
         mape(yte, linhas_baseline["C, coeficiente fixo da LDC"]))),
    ("H3", "A arvore perde porque nao extrapola tendencia",
     "verdadeira",
     "teto de %.0f, com %d dos %d meses de teste acima dele" % (teto, acima_teto, N_TESTE)),
    ("H4", "Alvo em razao devolve a arvore a disputa",
     "verdadeira",
     "MAPE da arvore cai de %.2f%% em nivel para %.2f%% em razao" % (
         mape(yte, p_arvore), mape(yte, p_arvore_razao))),
]

for codigo, hipotese, veredito, evidencia in quadro:
    print("%s: %s" % (codigo, hipotese))
    print("  veredito: %s" % veredito)
    print("  evidencia: %s" % evidencia)
    print()

H1: A granularidade do case e trimestral porque a fonte so tem trimestre
  veredito: falsa
  evidencia: a soma dos tres meses de cada trimestre reconcilia sem divergencia nas tres series de abate: 117 trimestres conferidos por serie, 0 divergentes no total

H2: A arvore captura o que a reta nao captura, logo vence
  veredito: falsa
  evidencia: MAPE 7.66% da arvore contra 4.25% da reta e 3.71% da baseline C

H3: A arvore perde porque nao extrapola tendencia
  veredito: verdadeira
  evidencia: teto de 1090166234, com 23 dos 24 meses de teste acima dele

H4: Alvo em razao devolve a arvore a disputa
  veredito: verdadeira
  evidencia: MAPE da arvore cai de 7.66% em nivel para 3.86% em razao

